In [2]:
# Fix duplicate upload - clear and reload the transactions table
from sqlalchemy import text

print("Clearing duplicate data from transactions table...")

with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE transactions"))
    conn.commit()

print("Table cleared. Reloading correct data...")

# Reload from CSV
transactions = pd.read_csv(r"C:\Users\Asus\Downloads\archive\Retail_Data_Transactions.csv")
transactions["customer_id"]  = transactions["customer_id"].astype(str).str.strip()
transactions["trans_date"]   = pd.to_datetime(transactions["trans_date"], format="%d-%b-%y", errors="coerce")
transactions["tran_amount"]  = pd.to_numeric(transactions["tran_amount"], errors="coerce")
transactions.dropna(subset=["customer_id", "trans_date", "tran_amount"], inplace=True)
transactions = transactions[transactions["tran_amount"] > 0]

transactions.to_sql("transactions", engine, if_exists="append", index=False, chunksize=1000)

# Verify
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM transactions")).scalar()
    print(f"transactions : {count:,} rows in DB")

print("Done. Correct row count restored.")

Clearing duplicate data from transactions table...
Table cleared. Reloading correct data...
transactions : 125,000 rows in DB
Done. Correct row count restored.


In [3]:
# STEP 2: OUTLIER DETECTION IN tran_amount
print("Outlier Analysis - tran_amount:")
print(transactions["tran_amount"].describe())

# IQR Method to detect outliers
Q1  = transactions["tran_amount"].quantile(0.25)
Q3  = transactions["tran_amount"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = transactions[
    (transactions["tran_amount"] < lower_bound) |
    (transactions["tran_amount"] > upper_bound)
]

print(f"\nIQR Lower Bound : {lower_bound:.2f}")
print(f"IQR Upper Bound : {upper_bound:.2f}")
print(f"Total Outliers  : {len(outliers):,} rows")
print(f"Outlier %       : {(len(outliers)/len(transactions))*100:.2f}%")

if len(outliers) > 0:
    print("\nSample outliers:")
    print(outliers.head(5).to_string(index=False))

Outlier Analysis - tran_amount:
count    125000.000000
mean         64.991912
std          22.860006
min          10.000000
25%          47.000000
50%          65.000000
75%          83.000000
max         105.000000
Name: tran_amount, dtype: float64

IQR Lower Bound : -7.00
IQR Upper Bound : 137.00
Total Outliers  : 0 rows
Outlier %       : 0.00%


In [7]:
# Extract month and year from trans_date
transactions["trans_month"] = transactions["trans_date"].dt.month
transactions["trans_year"]  = transactions["trans_date"].dt.year

# STEP 3: CREATE CALCULATED FIELDS
# Total spend per customer
customer_summary = transactions.groupby("customer_id").agg(
    total_spend       = ("tran_amount", "sum"),
    avg_transaction   = ("tran_amount", "mean"),
    transaction_count = ("tran_amount", "count"),
    first_purchase    = ("trans_date",  "min"),
    last_purchase     = ("trans_date",  "max")
).reset_index()

# Customer lifespan in days
customer_summary["lifespan_days"] = (
    customer_summary["last_purchase"] - customer_summary["first_purchase"]
).dt.days

# Round numeric columns
customer_summary["total_spend"]     = customer_summary["total_spend"].round(2)
customer_summary["avg_transaction"] = customer_summary["avg_transaction"].round(2)

print("Customer Summary Table:")
print(customer_summary.head(10).to_string(index=False))
print(f"\nTotal unique customers: {len(customer_summary):,}")

# Monthly sales summary
monthly_summary = transactions.groupby(["trans_year", "trans_month"]).agg(
    total_sales     = ("tran_amount", "sum"),
    total_orders    = ("tran_amount", "count"),
    avg_order_value = ("tran_amount", "mean")
).reset_index()

monthly_summary["avg_order_value"] = monthly_summary["avg_order_value"].round(2)

print("\nMonthly Sales Summary:")
print(monthly_summary.head(10).to_string(index=False))

Customer Summary Table:
customer_id  total_spend  avg_transaction  transaction_count first_purchase last_purchase  lifespan_days
     CS1112         1012            67.47                 15     2011-06-15    2015-01-14           1309
     CS1113         1490            74.50                 20     2011-05-27    2015-02-09           1354
     CS1114         1432            75.37                 19     2011-07-14    2015-02-12           1309
     CS1115         1659            75.41                 22     2011-08-10    2015-03-05           1303
     CS1116          857            65.92                 13     2011-06-27    2014-08-25           1155
     CS1117         1185            69.71                 17     2011-05-20    2014-07-02           1139
     CS1118         1011            67.40                 15     2011-05-18    2015-03-14           1396
     CS1119         1158            77.20                 15     2012-02-28    2015-03-05           1101
     CS1120         1677       

In [5]:
# Extract month and year from trans_date in Python
transactions["trans_month"] = transactions["trans_date"].dt.month
transactions["trans_year"]  = transactions["trans_date"].dt.year

# Monthly sales summary
monthly_summary = transactions.groupby(["trans_year", "trans_month"]).agg(
    total_sales     = ("tran_amount", "sum"),
    total_orders    = ("tran_amount", "count"),
    avg_order_value = ("tran_amount", "mean")
).reset_index()

monthly_summary["avg_order_value"] = monthly_summary["avg_order_value"].round(2)
monthly_summary["total_sales"]     = monthly_summary["total_sales"].round(2)

print("Monthly Sales Summary:")
print(monthly_summary.head(12).to_string(index=False))
print(f"\nTotal months in dataset: {len(monthly_summary)}")

Monthly Sales Summary:
 trans_year  trans_month  total_sales  total_orders  avg_order_value
       2011            5        98951          1485            66.63
       2011            6       174527          2707            64.47
       2011            7       178097          2726            65.33
       2011            8       188631          2914            64.73
       2011            9       169173          2605            64.94
       2011           10       182634          2839            64.33
       2011           11       166921          2570            64.95
       2011           12       181405          2812            64.51
       2012            1       177987          2737            65.03
       2012            2       170135          2619            64.96
       2012            3       180453          2743            65.79
       2012            4       168000          2613            64.29

Total months in dataset: 47


In [6]:
# STEP 4: SAVE CALCULATED TABLES TO SQL SERVER

print("Saving customer_summary to SQL Server...")
customer_summary.to_sql(
    "customer_summary",
    engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print(f"  customer_summary saved : {len(customer_summary):,} rows")

print("Saving monthly_summary to SQL Server...")
monthly_summary.to_sql(
    "monthly_summary",
    engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print(f"  monthly_summary saved  : {len(monthly_summary):,} rows")

# VERIFY
with engine.connect() as conn:
    cs = conn.execute(text("SELECT COUNT(*) FROM customer_summary")).scalar()
    ms = conn.execute(text("SELECT COUNT(*) FROM monthly_summary")).scalar()
    print(f"\nVerification:")
    print(f"  customer_summary in DB : {cs:,} rows")
    print(f"  monthly_summary in DB  : {ms:,} rows")

print("\nPhase 2 complete. Data is cleaned and prepared for analysis.")

Saving customer_summary to SQL Server...
  customer_summary saved : 6,889 rows
Saving monthly_summary to SQL Server...
  monthly_summary saved  : 47 rows

Verification:
  customer_summary in DB : 6,889 rows
  monthly_summary in DB  : 47 rows

Phase 2 complete. Data is cleaned and prepared for analysis.
